# 1. Define Data And Snapshot

Explore the raw Home Credit source, make durable project decisions in `config.py`, preview the configured pipeline, then create a logged immutable dataset.

## 1. Resolve Project Folder

Select the project folder from the notebook location. This does not require the project config to be final.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

import automl
from automl import data
from automl.data import FeatureRegistry
from automl.data.sources import LocalCSVSource

DRY_RUN = True


## 2. Choose A Candidate Source

Supported source patterns are ordinary Python objects. Keep one active source for the preview.

```python
# Local CSV
# source = LocalCSVSource(csv_path=config.project_dir / "data/application_train_sample.csv", unique_key="SK_ID_CURR")

# GCS parquet
# source = GCSParquetSource(gcs_uri="gs://bucket/path/application_train.parquet", unique_key="SK_ID_CURR")

# Snowflake query
# source = SnowflakeSource(base_table="TRAINING_BASE", base_table_sql="data/queries/base_table.sql", training_data_sql="data/queries/training_data.sql", unique_key="TRANSACTION_ID")
```

In [ ]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)


## 3. Inspect Raw Columns

Look at raw columns before committing project config decisions.

In [ ]:
source = LocalCSVSource(
    csv_path=config.project_dir / "data" / "application_train_sample.csv",
    unique_key="SK_ID_CURR",
)
source_df = source.load(project_dir=config.project_dir, nrows=1000)
source_df.head()


In [ ]:
raw_overview = {
    "shape": source_df.shape,
    "dtypes": source_df.dtypes.astype(str).to_frame("dtype"),
    "missingness": source_df.isna().mean().sort_values(ascending=False).to_frame("null_rate"),
    "target_distribution": source_df["TARGET"].value_counts(dropna=False).to_frame("rows"),
}
raw_overview


In [ ]:
unique_key_cols = ["SK_ID_CURR"]
pd.Series(
    {
        "unique_key": unique_key_cols,
        "duplicate_key_rows": int(source_df.duplicated(subset=unique_key_cols, keep=False).sum()),
        "null_key_rows": int(source_df[unique_key_cols].isna().any(axis=1).sum()),
    }
)


## 4. Draft Feature Registry Decisions

Use the registry interactively to review target, metadata, excluded columns, and initial feature roles before editing `config.py`.

In [ ]:
metadata_candidates = [col for col in source_df.columns if "ID" in col.upper() or col.upper().endswith("SK_ID_CURR")]
metadata_candidates


## 5. Edit `config.py`

Open and edit the durable project definition directly. Do not generate config source from the notebook.

Review these objects in `config.py`:

- `TASK`
- `DATA`
- `EVAL`
- `RUN_CONFIG`
- source block
- metadata and excluded columns
- dry-run row limit/default dry-run routing

In [ ]:
data_spec = config.require_data_spec()
registry = FeatureRegistry().build_from_df(
    source_df,
    target_column=config.raw_target_column,
    metadata_cols=data_spec.metadata_cols,
    exclude_cols=data_spec.exclude_cols,
)
registry.add_comment("SK_ID_CURR", "metadata: application identifier, not a model feature")
registry.to_dataframe()


## 6. Load Config And Preview Pipeline Output

After editing `config.py`, load the typed project and preview the configured pipeline locally. Previewing does not log MLflow or GCS artifacts.

In [ ]:
config.config_path


## 7. Explore Previewed Pipeline Data

Review the post-config train/test split view, registry, split report, target distribution, and normalized dtypes. Loop back to `config.py` if anything is wrong.

In [ ]:
loaded = data.build_dataset(session=active)
run_config = active.config.require_run_config()
train_predicate = run_config.splits.resolve(run_config.train_split)
holdout_predicate = run_config.splits.resolve(run_config.eval_split)
train_preview = loaded.df[train_predicate.mask(loaded.df)].reset_index(drop=True)
holdout_preview = loaded.df[holdout_predicate.mask(loaded.df)].reset_index(drop=True)
loaded


In [ ]:
train_preview.head()


In [ ]:
holdout_preview.head()


In [ ]:
loaded.registry.to_dataframe()


In [ ]:
{
    "train_split": run_config.train_split,
    "train_predicate": repr(train_predicate),
    "eval_split": run_config.eval_split,
    "eval_predicate": repr(holdout_predicate),
    "split_pct_col": loaded.dataset.split_pct_col,
}


In [ ]:
train_preview[loaded.dataset.target_column].value_counts(dropna=False).to_frame("train_rows")


## 8. Create Logged Dataset

Run this only when the previewed pipeline matches the intended contract. This writes the full immutable dataset parquet, feature registry, data manifest, dataset index, active dataset pointer, and source trace artifacts.

In [ ]:
loaded.df.dtypes.astype(str).to_frame("dtype")


## 9. Inspect Snapshot Result

Record the stable IDs and artifact pointers. Later notebooks should reload these artifacts instead of rebuilding the pipeline.

In [ ]:
dataset = data.materialize(session=active)
materialized = dataset.dataset
dataset_summary = {
    "dataset_id": materialized.id,
    "identity_hash": materialized.identity_hash,
    "data_uri": materialized.data_gcs_uri,
    "feature_registry_uri": materialized.registry_gcs_uri,
    "record_uri": materialized.record_uri,
    "source_identity_hash": materialized.component_hashes.source_identity,
    "dataset_content_hash": materialized.component_hashes.data_content,
    "schema_hash": materialized.component_hashes.schema,
}
dataset_summary